# Person 1 - Data Preprocessing and Feature Engineering

This notebook loads the agricultural dataset, audits it, creates date-based features, builds a preprocessing pipeline, and exports processed train/test files for the modeling team.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
file_path = 'Agri_yield_prediction.csv'
df = pd.read_csv(file_path)

print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())

In [ ]:
print('Data types:')
print(df.dtypes)

print('\nMissing values per column:')
print(df.isnull().sum())

print('\nNumber of duplicate rows:', df.duplicated().sum())

In [ ]:
df['Planting_Date'] = pd.to_datetime(df['Planting_Date'], errors='coerce')
df['Harvest_Date'] = pd.to_datetime(df['Harvest_Date'], errors='coerce')

df['Crop_Duration'] = (df['Harvest_Date'] - df['Planting_Date']).dt.days
df['Planting_Month'] = df['Planting_Date'].dt.month
df['Harvest_Month'] = df['Harvest_Date'].dt.month

df = df.drop(columns=['Planting_Date', 'Harvest_Date'])

print(df[['Crop_Duration', 'Planting_Month', 'Harvest_Month']].head())

In [ ]:
target_col = 'Yield'
X = df.drop(columns=[target_col])
y = df[target_col]

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print('Categorical columns:')
print(categorical_cols)

print('\nNumerical columns:')
print(numerical_cols)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

print(preprocessor)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train shape:', X_train.shape, y_train.shape)
print('Test shape:', X_test.shape, y_test.shape)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Processed train shape:', X_train_processed.shape)
print('Processed test shape:', X_test_processed.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, 'toarray') else X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed.toarray() if hasattr(X_test_processed, 'toarray') else X_test_processed,
    columns=feature_names
)

print('Total processed features:', len(feature_names))
X_train_processed_df.head()

In [ ]:
X_train_processed_df.to_csv('X_train_processed.csv', index=False)
X_test_processed_df.to_csv('X_test_processed.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print('Saved processed train/test files successfully.')

## What this notebook does

- Loads the agricultural dataset
- Checks columns, data types, missing values, and duplicates
- Converts planting and harvest dates into usable features
- Creates `Crop_Duration`, `Planting_Month`, and `Harvest_Month`
- Splits the data into features and target (`Yield`)
- Identifies numeric and categorical columns
- Fills missing values
- Scales numeric features
- One-hot encodes categorical features
- Creates train/test sets
- Saves processed files for the modeling teammates